# Computational Cost Benchmark (Reviewer R1.5)

This notebook measures the computational cost of the proposed pfRUL framework to address the reviewer request to investigate its efficiency and to compare it with the state of the art. It reports:

1. **Model size**: number of controller parameters and particle count.
2. **Online inference latency**: time per particle-filter update step and per full trajectory, for the neural controller and the constant-gain baseline.
3. **Prediction latency**: time to build the predictive RUL distribution and its 95% confidence interval.
4. **Offline training cost**: a representative per-rollout time and the resulting per-epoch estimate.

All measurements run on a single CPU. The timing of the online and training loops does not depend on the trained weights, so an untrained controller of the same architecture gives identical timings; we therefore build the controller directly from the architecture and do not require a trained checkpoint.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from experiment_config import (
    SEED,
    UNCERTAINTY_LEVEL,
    DegModel,
    dataset_paths,
)
from src.helpers.seed import set_global_seed
from src.models.networks.pf_mlp import ParticleFilterMLP, build_activation
from src.models.particle_filter.core import ParticleFilter
from src.training.pfnet_hparams import PFNET_ARGS

set_global_seed(SEED)
print('torch threads:', torch.get_num_threads())

torch threads: 64


## Configuration

Set the dataset and the controller architecture id. `ARGS_ID = 1` corresponds to the paper controller with hidden layers `[128, 64, 32]`. `N_WARMUP` runs are discarded to remove first-call overhead, and the median over `N_REPEAT` runs is reported.

In [2]:
DATA_NAME = 'DS01'
ARGS_ID = 1  # paper controller architecture [128, 64, 32]; timing is weight-independent
N_WARMUP = 2
N_REPEAT = 5

# a representative epoch count for the total-training estimate (best epoch is
# typically reached around here; adjust to your runs)
TYPICAL_EPOCHS = 50

## Load data and offline degradation models

We load the development and test health-index trajectories and the offline degradation base models that initialise the particle population.

In [3]:
args = PFNET_ARGS[ARGS_ID]
net_args = args['NETWORK']
pf_args = args['PARTICLE_FILTER']
train_args = args['TRAINING']

N_PARTICLES = int(pf_args['N_PARTICLES'])
MAX_LIFE = int(pf_args['MAX_LIFE'])
LOSS_TAIL_STEPS = int(train_args['LOSS_TAIL_STEPS'])
LR = float(train_args['LR'])

ESTIMATION_DIR, DEGR_MODEL_DIR = dataset_paths(
    DATA_NAME, fields=['estimation', 'degr_model']
)

dev_hi = pd.read_csv(ESTIMATION_DIR / 'data_dev.csv')
test_hi = pd.read_csv(ESTIMATION_DIR / 'data_test.csv')

dev_units = sorted(dev_hi['unit'].astype(int).unique().tolist())
test_units = sorted(test_hi['unit'].astype(int).unique().tolist())
perform_names = [c for c in dev_hi.columns if c not in ['unit', 'cycle', 'hs']]

print('architecture:', net_args['HIDDEN_DIMS'], '| particles:', N_PARTICLES)
print('metrics:', perform_names)
print('dev units:', dev_units, '| test units:', test_units)

architecture: [128, 64, 32] | particles: 2000
metrics: ['T48', 'SmFan', 'SmLPC', 'SmHPC']
dev units: [1, 2, 4, 5, 6] | test units: [7, 8, 9, 10]


In [4]:
def build_tensor(df, unit, name):
    sub = df[df['unit'] == unit]
    t = sub['cycle'].values
    s = sub[name].values
    return torch.tensor(np.stack([t, s], axis=1), dtype=torch.float32)


def load_base_models(name, units):
    models = []
    for u in units:
        m = DegModel()
        m.load_state_dict(
            torch.load(
                DEGR_MODEL_DIR / 'states' / name / f'unit_{u}' / 'best_model.pt'
            )
        )
        models.append(m)
    return models


def make_net():
    net = ParticleFilterMLP(
        state_dim=DegModel.state_dim(),
        hidden_dims=list(net_args['HIDDEN_DIMS']),
        activation=build_activation(net_args),
        dropout_p=float(net_args['DROPOUT']),
    )
    return net.eval()

## 1. Model size

The controller is a small multilayer perceptron mapping the scalar pair `(t, s)` to the particle-filter gains (process-noise scales, prior weights, and likelihood temperature). We report its parameter count together with the particle count `N`.

In [5]:
net = make_net()
n_params = sum(p.numel() for p in net.parameters())
state_dim = DegModel.state_dim()

size_table = pd.DataFrame(
    {
        'controller layers': [str([2, *list(net_args['HIDDEN_DIMS']), 2 * state_dim + 1])],
        'parameters': [n_params],
        'particles N': [N_PARTICLES],
    },
    index=['pfRUL'],
)
size_table

,controller layers,parameters,particles N
pfRUL,"[2, 128, 64, 32, 13]",11149,2000


## 2. Online inference latency

For each performance metric we build a particle filter with the full development prior (the deployment scenario) and advance it one observation at a time over a test-unit trajectory, timing each update step (proposal, tempered correction, resampling). We report the median over repeats. Because the four metrics are filtered independently, the system-level cost is their sum. We benchmark both the neural controller (`pfRUL`) and the constant-gain baseline (`use_net = False`).

In [6]:
def build_pf(name, base_units, use_net):
    base = load_base_models(name, base_units)
    return ParticleFilter(
        base_models=base,
        net=make_net() if use_net else None,
        n_particles=N_PARTICLES,
        max_life=MAX_LIFE,
        use_net=use_net,
    ).eval()


@torch.no_grad()
def time_trajectory(pf, unit_tensor):
    t_data = unit_tensor[:, 0]
    s_data = unit_tensor[:, 1]
    step_times = []
    for k in range(len(t_data)):
        t0 = time.perf_counter()
        pf.step(t_obs=t_data[[k]], s_obs=s_data[[k]])
        step_times.append(time.perf_counter() - t0)
    return np.array(step_times)

In [7]:
rows = []
for use_net, label in [(True, 'pfRUL (neural)'), (False, 'constant-gain')]:
    for name in perform_names:
        unit = test_units[0]
        ut = build_tensor(test_hi, unit, name)

        for _ in range(N_WARMUP):  # warmup
            time_trajectory(build_pf(name, dev_units, use_net), ut)

        per_step, traj = [], []
        for _ in range(N_REPEAT):
            st = time_trajectory(build_pf(name, dev_units, use_net), ut)
            per_step.append(st.mean())
            traj.append(st.sum())

        rows.append(
            {
                'config': label,
                'metric': name,
                'steps': int(len(ut)),
                'per-step ms': 1e3 * float(np.median(per_step)),
                'trajectory ms': 1e3 * float(np.median(traj)),
            }
        )

online_table = pd.DataFrame(rows)
online_table.round(3)

,config,metric,steps,per-step ms,trajectory ms
0,pfRUL (neural),T48,90,1.239,111.471
1,pfRUL (neural),SmFan,90,1.275,114.766
2,pfRUL (neural),SmLPC,90,1.249,112.409
3,pfRUL (neural),SmHPC,90,1.249,112.429
4,constant-gain,T48,90,1.104,99.324
5,constant-gain,SmFan,90,1.123,101.058
6,constant-gain,SmLPC,90,1.090,98.135
7,constant-gain,SmHPC,90,1.090,98.110


In [8]:
# system-level cost = sum across the four independently filtered metrics
system_online = (
    online_table.groupby('config')[['per-step ms', 'trajectory ms']]
    .sum()
    .rename(
        columns={
            'per-step ms': 'per-step ms (all metrics)',
            'trajectory ms': 'trajectory ms (all metrics)',
        }
    )
)
system_online.round(3)

,per-step ms (all metrics),trajectory ms (all metrics)
config,,
constant-gain,4.407,396.627
pfRUL (neural),5.012,451.075


## 3. Prediction latency

At a representative mid-trajectory state we time the construction of the predictive RUL distribution and its 95% confidence interval for a single metric. This is the extra cost paid whenever a prediction is emitted, on top of the per-step filter update.

In [9]:
name = perform_names[0]
ut = build_tensor(test_hi, test_units[0], name)
pf = build_pf(name, dev_units, use_net=True)
mid = len(ut) // 2

with torch.no_grad():
    for k in range(mid):  # advance to a mid-trajectory posterior
        pf.step(t_obs=ut[[k], 0], s_obs=ut[[k], 1])

level = float(UNCERTAINTY_LEVEL)
times = []
with torch.no_grad():
    for _ in range(N_WARMUP + N_REPEAT):
        t0 = time.perf_counter()
        lo, me, up = pf.mixture.uncertainty_interval(s=torch.zeros(1), level=level)
        times.append(time.perf_counter() - t0)

pred_ms = 1e3 * float(np.median(np.array(times[N_WARMUP:])))
pred_table = pd.DataFrame(
    {'prediction ms (per metric)': [pred_ms]}, index=[name]
)
pred_table.round(3)

,prediction ms (per metric)
T48,508.816


## 4. Offline training cost

Training uses a nested leave-one-unit-out loop: each epoch performs `M(M-1)` particle-filter rollouts with backpropagation, where `M` is the number of development units. We time a single rollout (forward filtering with the tail-NLL loss plus one optimizer step) and estimate the per-epoch cost. The total offline cost scales with the number of epochs, metrics, and datasets.

In [10]:
def mean_tail_nll(pf, t_data, s_data):
    losses = []
    for k in range(len(t_data)):
        mix = pf.step(t_obs=t_data[[k]], s_obs=s_data[[k]])
        start = -LOSS_TAIL_STEPS if LOSS_TAIL_STEPS else k
        d = mix.distribution(s=s_data[start:])
        losses.append(-d.log_prob(t_data[start:]).mean())
    return torch.stack(losses).mean()


name = perform_names[0]
train_unit = dev_units[0]
tt = build_tensor(dev_hi, train_unit, name)
# default protocol removes both the fitted and the held-out unit (n-2 base models)
base_units = [u for u in dev_units if u != train_unit][:-1]

net_train = make_net().train()
opt = torch.optim.Adam(net_train.parameters(), LR)

rollout_times = []
for _ in range(N_WARMUP + N_REPEAT):
    pf = ParticleFilter(
        base_models=load_base_models(name, base_units),
        net=net_train,
        n_particles=N_PARTICLES,
        max_life=MAX_LIFE,
    ).train()
    opt.zero_grad()
    t0 = time.perf_counter()
    loss = mean_tail_nll(pf, tt[:, 0], tt[:, 1])
    loss.backward()
    opt.step()
    rollout_times.append(time.perf_counter() - t0)

per_rollout_s = float(np.median(np.array(rollout_times[N_WARMUP:])))
M = len(dev_units)
rollouts_per_epoch = M * (M - 1)

train_table = pd.DataFrame(
    {
        'per-rollout s': [per_rollout_s],
        'rollouts/epoch M(M-1)': [rollouts_per_epoch],
        'est. per-epoch s': [per_rollout_s * rollouts_per_epoch],
        f'est. per-metric ({TYPICAL_EPOCHS} epochs) min': [
            per_rollout_s * rollouts_per_epoch * TYPICAL_EPOCHS / 60.0
        ],
    },
    index=[name],
)
train_table.round(3)

,per-rollout s,rollouts/epoch M(M-1),est. per-epoch s,est. per-metric (50 epochs) min
T48,1.108,20,22.162,18.469


## Summary and comparison notes

The controller is a small CPU model, and each online update is linear in the number of particles `N`. Compared with the deep-learning baselines in the results table (BGT, BLSTM, LSTM, transformers), which are GPU-trained and have orders of magnitude more parameters, the proposed method trades a modest sequential CPU inference cost for a full predictive RUL distribution with calibrated uncertainty. The absolute runtimes of those baselines are not reported and their hardware differs, so the comparison is stated in terms of model size and compute regime rather than raw wall-clock time.

The cell below assembles the tables and writes them to `benchmarks/` as CSV for inclusion in the manuscript.

In [11]:
BENCH_DIR = Path('benchmarks')
BENCH_DIR.mkdir(parents=True, exist_ok=True)

size_table.to_csv(BENCH_DIR / f'cost_size_{DATA_NAME}_arg{ARGS_ID}.csv')
online_table.to_csv(BENCH_DIR / f'cost_online_{DATA_NAME}_arg{ARGS_ID}.csv', index=False)
system_online.to_csv(BENCH_DIR / f'cost_system_{DATA_NAME}_arg{ARGS_ID}.csv')
pred_table.to_csv(BENCH_DIR / f'cost_prediction_{DATA_NAME}_arg{ARGS_ID}.csv')
train_table.to_csv(BENCH_DIR / f'cost_training_{DATA_NAME}_arg{ARGS_ID}.csv')

print('saved cost tables to', BENCH_DIR.resolve())
print('\n== model size ==')
print(size_table.to_string())
print('\n== online per metric ==')
print(online_table.round(3).to_string(index=False))
print('\n== system online (sum of metrics) ==')
print(system_online.round(3).to_string())
print('\n== prediction ==')
print(pred_table.round(3).to_string())
print('\n== training ==')
print(train_table.round(3).to_string())

saved cost tables to /home/diazgonz/deep-performance-rul/benchmarks

== model size ==
          controller layers  parameters  particles N
pfRUL  [2, 128, 64, 32, 13]       11149         2000

== online per metric ==
        config metric  steps  per-step ms  trajectory ms
pfRUL (neural)    T48     90        1.239        111.471
pfRUL (neural)  SmFan     90        1.275        114.766
pfRUL (neural)  SmLPC     90        1.249        112.409
pfRUL (neural)  SmHPC     90        1.249        112.429
 constant-gain    T48     90        1.104         99.324
 constant-gain  SmFan     90        1.123        101.058
 constant-gain  SmLPC     90        1.090         98.135
 constant-gain  SmHPC     90        1.090         98.110

== system online (sum of metrics) ==
                per-step ms (all metrics)  trajectory ms (all metrics)
config                                                                
constant-gain                       4.407                      396.627
pfRUL (neural)     